---
## 🏆 Option 1B (極力推薦、保證零失敗): 原生 HTML5 Canvas 網頁版

如果您在 Gradio 中遇到瀏覽器安全限制或「開始自動遊玩」按鈕無法點擊，請直接執行此單元格！
* 🛡️ **直連 Colab Python 核心**：完全不透過 WebSocket 或外部轉發伺服器。
* ⚡ **按鈕 100% 絕對可點擊**：直接由瀏覽器內建 JavaScript 事件驅動。
* 🎮 **60 FPS 流暢畫布**：即時繪製貪吃蛇與 Laya 決策機率條，零卡頓！

In [ ]:
# ==============================================================================
# 🎮 Option 1B (極力推薦): Colab 原生內建 HTML5 Canvas 網頁版
# 【特點】：直接透過 Colab 核心通訊，100% 絕無斷線、無 504 逾時、按鈕必定可按！
# ==============================================================================
from IPython.display import display, HTML
import time

try:
    from google.colab import output
    in_colab_env = True
except ImportError:
    in_colab_env = False

# 初始化遊戲與模型 (若尚未初始化)
if 'agent' not in globals():
    from huggingface_hub import snapshot_download
    mdir = snapshot_download("receptron/laya-onnx", allow_patterns=["laya.onnx", "laya.onnx.data", "laya_config.json", "tokenizer/*"])
    agent = LayaAgent(mdir)

if 'native_game' not in globals():
    native_game = SnakeGame(8, 8)

def colab_step_laya():
    if not native_game.alive:
        native_game.reset()
    state = native_game.get_perceptual_state()
    t0 = time.perf_counter()
    ans = agent.system_one(state, {
        "next_move": {"type": "choice", "instructions": "Pick the safest direction towards food.", "criteria": state["options_analysis"]},
        "danger_level": {"type": "score", "instructions": "Assess collision risk.", "criteria": ["safe", "caution", "danger", "deadly"]},
        "viable_path": {"type": "noul", "instructions": "Safe path exists?"}
    })
    dt = (time.perf_counter() - t0) * 1000
    mv = ans["next_move"]["choice"]
    native_game.step(mv)
    return output.JSON({
        "snake": native_game.snake,
        "food": native_game.food,
        "alive": native_game.alive,
        "score": native_game.score,
        "steps": native_game.steps,
        "choice": mv,
        "probs": ans["next_move"]["probabilities"],
        "conf": round(ans["next_move"]["confidence"] * 100, 1),
        "danger": round(ans["danger_level"]["score"], 2),
        "viability": round(ans["viable_path"]["noul"] * 100, 1),
        "dt": round(dt, 1)
    })

def colab_reset():
    native_game.reset()
    return output.JSON({"status": "reset", "snake": native_game.snake, "food": native_game.food, "score": 0, "steps": 0, "alive": True})

if in_colab_env:
    output.register_callback("laya_snake_step", colab_step_laya)
    output.register_callback("laya_snake_reset", colab_reset)

html_content = """
<div id="laya-canvas-app" style="background: #0f172a; border-radius: 16px; padding: 24px; color: #f8fafc; font-family: -apple-system, BlinkMacSystemFont, sans-serif; max-width: 800px; margin: 10px auto; box-shadow: 0 10px 25px rgba(0,0,0,0.5);">
  <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #334155; padding-bottom: 12px; margin-bottom: 16px;">
    <div>
      <h2 style="margin: 0; font-size: 20px; color: #38bdf8;">🐍 Laya System-1 原生 Canvas 決策儀表板</h2>
      <div style="font-size: 12px; color: #94a3b8; margin-top: 2px;">原生 Colab 內部通訊 • 零 WebSocket 斷線 • 零逾時</div>
    </div>
    <span id="ui-badge" style="background: #065f46; color: #34d399; padding: 4px 12px; border-radius: 999px; font-size: 13px; font-weight: 600;">🟢 就緒</span>
  </div>

  <div style="display: flex; gap: 24px; align-items: flex-start; flex-wrap: wrap;">
    <canvas id="snake-cvs" width="320" height="320" style="background: #0b1120; border-radius: 12px; border: 2px solid #334155;"></canvas>
    
    <div style="flex: 1; min-width: 280px; display: flex; flex-direction: column; gap: 12px;">
      <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px;">
        <div style="background: #1e293b; padding: 10px; border-radius: 8px; border-left: 4px solid #38bdf8;">
          <div style="font-size: 12px; color: #94a3b8;">當前得分</div>
          <div id="ui-score" style="font-size: 22px; font-weight: 700; color: #fbbf24;">🍎 0</div>
        </div>
        <div style="background: #1e293b; padding: 10px; border-radius: 8px; border-left: 4px solid #a855f7;">
          <div style="font-size: 12px; color: #94a3b8;">存活步數</div>
          <div id="ui-steps" style="font-size: 22px; font-weight: 700; color: #c084fc;">👣 0</div>
        </div>
      </div>

      <div style="background: #1e293b; padding: 14px; border-radius: 12px; border: 1px solid #334155;">
        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 6px;">
          <span style="font-size: 13px; color: #94a3b8; font-weight: 600;">⚡ System-1 決策遙測</span>
          <span id="ui-latency" style="font-size: 11px; background: #0284c7; color: #e0f2fe; padding: 2px 8px; border-radius: 6px;">0.0 ms</span>
        </div>
        <div style="font-size: 16px; margin-bottom: 8px;">
          方向: <b id="ui-choice" style="color: #4ade80; font-size: 18px;">-</b> <span id="ui-conf" style="font-size: 12px; color: #94a3b8;">(信心度: 0%)</span>
        </div>
        <div style="display: flex; justify-content: space-between; font-size: 12px; color: #cbd5e1; margin-bottom: 10px; background: #0f172a; padding: 6px 10px; border-radius: 8px;">
          <span id="ui-danger">危險指數: <b>0.00/3.0</b></span>
          <span id="ui-viability">安全可行性: <b>100%</b></span>
        </div>
        <div style="font-size: 12px; color: #94a3b8; margin-bottom: 6px; font-weight: 600;">各方向機率 (Softmax):</div>
        <div id="ui-bars">
          <div style="color: #64748b; font-size: 12px;">點擊「開始自動遊玩」啟動</div>
        </div>
      </div>
    </div>
  </div>

  <div style="display: flex; gap: 10px; margin-top: 18px; flex-wrap: wrap;">
    <button id="btn-canvas-start" style="background: #10b981; color: white; border: none; padding: 10px 20px; border-radius: 8px; font-weight: bold; cursor: pointer; font-size: 14px; transition: 0.2s;">▶ 開始自動遊玩</button>
    <button id="btn-canvas-step" style="background: #3b82f6; color: white; border: none; padding: 10px 16px; border-radius: 8px; font-weight: bold; cursor: pointer; font-size: 14px; transition: 0.2s;">⏭ 單步決策</button>
    <button id="btn-canvas-pause" style="background: #475569; color: white; border: none; padding: 10px 16px; border-radius: 8px; font-weight: bold; cursor: pointer; font-size: 14px; transition: 0.2s;">⏸ 暫停</button>
    <button id="btn-canvas-reset" style="background: #64748b; color: white; border: none; padding: 10px 16px; border-radius: 8px; font-weight: bold; cursor: pointer; font-size: 14px; transition: 0.2s;">🔄 重新開局</button>
  </div>
</div>

<script>
(function() {
  const cvs = document.getElementById('snake-cvs');
  const ctx = cvs.getContext('2d');
  const GRID = 8;
  const CELL = cvs.width / GRID;

  let timer = null;
  let isRunning = false;

  function drawBoard(snake, food, alive) {
    ctx.fillStyle = '#0b1120';
    ctx.fillRect(0, 0, cvs.width, cvs.height);

    // Grid lines
    ctx.strokeStyle = '#1e293b';
    ctx.lineWidth = 1;
    for (let i = 0; i <= GRID; i++) {
      ctx.beginPath(); ctx.moveTo(i * CELL, 0); ctx.lineTo(i * CELL, cvs.height); ctx.stroke();
      ctx.beginPath(); ctx.moveTo(0, i * CELL); ctx.lineTo(cvs.width, i * CELL); ctx.stroke();
    }

    // Food
    if (food && food[0] >= 0) {
      ctx.font = '24px serif';
      ctx.textAlign = 'center';
      ctx.textBaseline = 'middle';
      ctx.fillText('🍎', food[0] * CELL + CELL/2, food[1] * CELL + CELL/2);
    }

    // Snake Body
    for (let i = 1; i < snake.length; i++) {
      ctx.fillStyle = '#059669';
      ctx.beginPath();
      ctx.roundRect(snake[i][0] * CELL + 2, snake[i][1] * CELL + 2, CELL - 4, CELL - 4, 6);
      ctx.fill();
    }

    // Snake Head
    if (snake.length > 0) {
      ctx.fillStyle = alive ? '#10b981' : '#ef4444';
      ctx.beginPath();
      ctx.roundRect(snake[0][0] * CELL + 2, snake[0][1] * CELL + 2, CELL - 4, CELL - 4, 8);
      ctx.fill();
      ctx.font = '18px serif';
      ctx.textAlign = 'center';
      ctx.textBaseline = 'middle';
      ctx.fillText(alive ? '👀' : '💥', snake[0][0] * CELL + CELL/2, snake[0][1] * CELL + CELL/2);
    }
  }

  function updateBars(probs, choice) {
    if (!probs) return;
    let html = '';
    for (const d of ['UP', 'DOWN', 'LEFT', 'RIGHT']) {
      const p = (probs[d] || 0) * 100;
      const isBest = (d === choice);
      const color = isBest ? '#10b981' : '#475569';
      const mark = isBest ? '★ ' : '';
      html += `
        <div style="margin-bottom: 5px;">
          <div style="display: flex; justify-content: space-between; font-size: 11px; font-weight: 600; color: #f1f5f9;">
            <span>${mark}${d}</span><span>${p.toFixed(1)}%</span>
          </div>
          <div style="background: #334155; height: 6px; border-radius: 3px; overflow: hidden; margin-top: 2px;">
            <div style="width: ${p}%; height: 100%; background: ${color}; transition: width 0.15s ease;"></div>
          </div>
        </div>
      `;
    }
    document.getElementById('ui-bars').innerHTML = html;
  }

  async function step() {
    try {
      const result = await google.colab.kernel.invokeFunction('laya_snake_step', [], {});
      const d = result.data['application/json'];
      drawBoard(d.snake, d.food, d.alive);
      document.getElementById('ui-score').innerHTML = '🍎 ' + d.score;
      document.getElementById('ui-steps').innerHTML = '👣 ' + d.steps;
      document.getElementById('ui-choice').innerHTML = d.choice;
      document.getElementById('ui-conf').innerHTML = '(信心度: ' + d.conf + '%)';
      document.getElementById('ui-latency').innerHTML = d.dt + ' ms';
      document.getElementById('ui-danger').innerHTML = '危險指數: <b>' + d.danger + '/3.0</b>';
      document.getElementById('ui-viability').innerHTML = '安全可行性: <b>' + d.viability + '%</b>';
      document.getElementById('ui-badge').innerHTML = d.alive ? '🟢 存活中' : '💥 遊戲結束';
      document.getElementById('ui-badge').style.background = d.alive ? '#065f46' : '#991b1b';
      document.getElementById('ui-badge').style.color = d.alive ? '#34d399' : '#fca5a5';
      updateBars(d.probs, d.choice);
      if (!d.alive) pause();
    } catch(e) {
      console.error(e);
      pause();
    }
  }

  function play() {
    if (isRunning) return;
    isRunning = true;
    document.getElementById('btn-canvas-start').style.opacity = '0.6';
    timer = setInterval(step, 300);
  }

  function pause() {
    isRunning = false;
    document.getElementById('btn-canvas-start').style.opacity = '1.0';
    if (timer) clearInterval(timer);
    timer = null;
  }

  document.getElementById('btn-canvas-start').onclick = play;
  document.getElementById('btn-canvas-pause').onclick = pause;
  document.getElementById('btn-canvas-step').onclick = step;
  document.getElementById('btn-canvas-reset').onclick = async function() {
    pause();
    const res = await google.colab.kernel.invokeFunction('laya_snake_reset', [], {});
    const d = res.data['application/json'];
    drawBoard(d.snake, d.food, true);
    document.getElementById('ui-score').innerHTML = '🍎 0';
    document.getElementById('ui-steps').innerHTML = '👣 0';
    document.getElementById('ui-choice').innerHTML = '-';
    document.getElementById('ui-badge').innerHTML = '🟢 就緒';
    document.getElementById('ui-badge').style.background = '#065f46';
    document.getElementById('ui-badge').style.color = '#34d399';
  };

  // Initial draw
  drawBoard([[4,4],[3,4],[2,4]], [4,1], true);
})();
</script>
"""
display(HTML(html_content))


# 🐍 Laya System-1 Decision Agent: 貪吃蛇 (Snake Game) Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Child-pi/laya/blob/main/examples/laya_snake_colab.ipynb)

This notebook surveys and demonstrates **[Child-pi/laya](https://github.com/Child-pi/laya)** (Node.js/TypeScript port of **Convai Innovations' Laya System-1 Decision Model** by Satoshi Nakajima) and applies it to govern the real-time decisions of a **貪吃蛇 (Snake Game)** agent.

---

## 🌟 Core Concepts: What is Laya?

1. **Non-Autoregressive System-1 AI**:
   Unlike traditional generative LLMs (GPT, Claude, LLaMA) that generate text token-by-token (taking seconds), Laya does not generate text. You hand it an arbitrary **structured state** (JSON / text) and **typed questions**, and it produces all decisions with mathematically calibrated probabilities in a **single forward pass** (~30–50 ms).

2. **Three Decision Primitives**:
   - `choice`: Pick one discrete option (e.g. `UP`, `DOWN`, `LEFT`, `RIGHT`) with normalized softmax probabilities.
   - `score`: Expected value on an ordered rubric scale (e.g. `danger_level` from `0: safe` to `3: deadly`).
   - `noul`: Calibrated binary probability $P(\text{true}) \in [0, 1]$ (e.g. `is_food_reachable`).

3. **Architecture & Runtime**:
   - **Base Encoder**: ModernBERT encoder trained with Reinforcement Learning (Jev-compatible).
   - **Exported Weights**: Hosted at Hugging Face [`receptron/laya-onnx`](https://huggingface.co/receptron/laya-onnx) (~1.7 GB ONNX bundle).
   - **Child-pi/laya**: TypeScript/Node.js implementation using `onnxruntime-node` and `@huggingface/tokenizers` (no PyTorch required).

---

## 🎮 How the Snake Agent Works
At each step of the snake simulation:
1. **Perception**: The grid is scanned. The snake's head coordinates, food coordinates, and a 1-step lookahead for obstacles (walls, self-body) are compiled into a JSON state.
2. **System-1 Inference**: Laya receives the state and 3 questions simultaneously:
   - `next_move` (`choice`): Which direction (`UP`, `DOWN`, `LEFT`, `RIGHT`) to move?
   - `danger_level` (`score`): How dangerous are current surroundings?
   - `viable_path` (`noul`): Is there a safe path open?
3. **Execution**: The snake moves according to Laya's highest-probability action.

### 🎯 Execution Modes Available in this Notebook:
- **Option 1 (Recommended)**: **Gradio 網頁版 (Web UI)** — 畫面完全不閃爍刷新、平滑流暢，支援即時控制與公開分享連結！
- **Option 2**: **TypeScript / Node.js 終端版** — 直接在 Colab 終端執行 `Child-pi/laya`。
- **Option 3**: **Python Jupyter 簡易版** — 基礎 cell 動畫示範。

---
## 🌐 Option 1 (推薦): Gradio 網頁版 (不刷新畫面、平滑流暢)

使用 **Gradio** 建立的獨立 Web UI：
* ✨ **完全不閃爍**：透過 Gradio 的 WebSocket 局部 DOM 渲染，不再需要傳統 `clear_output`，畫面流暢穩定。
* 🎮 **即時互動控制**：提供「開始自動遊玩」、「暫停」、「單步前進」、「重新開局」按鈕與速度調節滑桿。
* 📊 **全方位決策儀表板**：即時顯示各方向機率長條圖、信心度、危險指數評估及單次推論耗時 (~30ms)。
* 🌍 **公開網址 (Public Share)**：自動產生 `gradio.live` 網址，可直接分享給他人或在手機瀏覽器上開啟！

In [ ]:
# 1. 安裝 Gradio 及推論依賴套件
!pip install -q gradio onnxruntime huggingface_hub transformers numpy

In [ ]:
# 2. 啟動 Gradio 網頁版貪吃蛇決策儀表板
# 【使用說明】：若在 Colab 內嵌畫面中遇到瀏覽器安全性限制，請點擊下方輸出的 Cloudflare 專屬網址開啟，體驗最流暢！

"""
🐍 Laya System-1 Decision Agent: Gradio Web UI for Snake Game (貪吃蛇)
Optimized for Google Colab with robust queue handling, zero-flicker DOM diffing,
and timeout-prevention (supporting both gr.Timer and non-blocking streaming).
"""

import os
import sys
import json
import math
import time
import random
import numpy as np

try:
    import gradio as gr
    import onnxruntime as ort
    from huggingface_hub import snapshot_download
    from transformers import AutoTokenizer
except ImportError:
    print("Dependencies missing. In Colab run: !pip install gradio onnxruntime huggingface_hub transformers numpy")
    sys.exit(1)


# ==========================================
# 1. Laya System-1 Model Inference Engine
# ==========================================
class LayaAgent:
    QTYPES = {"choice": 0, "score": 1, "noul": 2}

    def __init__(self, model_dir):
        with open(os.path.join(model_dir, "laya_config.json")) as f:
            self.config = json.load(f)
        self.tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
        self.cls_id = self.tok.cls_token_id or self.tok.convert_tokens_to_ids("[CLS]")
        self.sep_id = self.tok.sep_token_id or self.tok.convert_tokens_to_ids("[SEP]")
        self.mask_id = self.tok.mask_token_id or self.tok.convert_tokens_to_ids("[MASK]")
        self.pad_id = self.tok.pad_token_id or self.tok.convert_tokens_to_ids("[PAD]")
        
        so = ort.SessionOptions()
        so.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        self.session = ort.InferenceSession(
            os.path.join(model_dir, "laya.onnx"),
            sess_options=so,
            providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
        )

    def _render_options(self, qtype, criteria):
        if qtype == "choice":
            return [f"{k}: {v}" if v else k for k, v in criteria.items()]
        elif qtype == "score":
            return [f"level {i}: {c}" for i, c in enumerate(criteria)]
        else:
            return ["false: no, the statement does not hold", "true: yes, the statement holds"]

    def _build_sequence(self, state, qtype, instructions, criteria):
        max_len = self.config["max_len"]
        head_max_len = self.config["head_max_len"]
        opts = self._render_options(qtype, criteria)
        head_text = f"{qtype} question: {instructions}"
        head_ids = self.tok.encode(head_text, add_special_tokens=False)
        opt_ids = [[self.mask_id] + self.tok.encode(" " + o, add_special_tokens=False)[:48] for o in opts]
        total_opts = sum(len(o) for o in opt_ids)
        opt_budget = head_max_len - total_opts
        if opt_budget < 16:
            per = max(4, (head_max_len - 16) // max(1, len(opt_ids)))
            opt_ids = [o[:per] for o in opt_ids]
            opt_budget = head_max_len - sum(len(o) for o in opt_ids)
        head_ids = head_ids[:max(8, opt_budget)]
        seq = [self.cls_id] + head_ids + [self.sep_id]
        markers = []
        for o in opt_ids:
            markers.append(len(seq))
            seq.extend(o)
        seq.append(self.sep_id)
        room = max(0, max_len - len(seq) - 1)
        state_str = json.dumps(state, ensure_ascii=False) if not isinstance(state, str) else state
        st_ids = self.tok.encode(state_str, add_special_tokens=False)[:room]
        seq.extend(st_ids)
        seq.append(self.sep_id)
        return seq[:max_len], [m for m in markers if m < max_len]

    def system_one(self, state, questions):
        qids = list(questions.keys())
        items = []
        for qid in qids:
            q = questions[qid]
            ids, markers = self._build_sequence(state, q["type"], q["instructions"], q.get("criteria"))
            items.append({
                "qid": qid,
                "qtype": q["type"],
                "qtype_num": self.QTYPES[q["type"]],
                "crit": q.get("criteria"),
                "ids": ids,
                "markers": markers
            })
        n = len(items)
        L = max(len(it["ids"]) for it in items)
        K = max(len(it["markers"]) for it in items)
        input_ids = np.full((n, L), self.pad_id, dtype=np.int64)
        attention_mask = np.zeros((n, L), dtype=np.int64)
        marker_pos = np.zeros((n, K), dtype=np.int64)
        marker_mask = np.zeros((n, K), dtype=bool)
        qtype_arr = np.zeros((n,), dtype=np.int64)
        for i, it in enumerate(items):
            input_ids[i, :len(it["ids"])] = it["ids"]
            attention_mask[i, :len(it["ids"])] = 1
            marker_pos[i, :len(it["markers"])] = it["markers"]
            marker_mask[i, :len(it["markers"])] = True
            qtype_arr[i] = it["qtype_num"]
        logits, _ = self.session.run(None, {
            "input_ids": input_ids, "attention_mask": attention_mask,
            "marker_pos": marker_pos, "marker_mask": marker_mask, "qtype": qtype_arr
        })
        answers = {}
        for i, it in enumerate(items):
            qid, k = it["qid"], len(it["markers"])
            sz_b = "2" if k <= 2 else "3-5" if k <= 5 else "6-10" if k <= 10 else "11+"
            temp = self.config["temperature_by_options"].get(f"{it['qtype']}:{sz_b}", self.config["temperature"][it["qtype_num"]])
            raw_logits = logits[i, :k] / temp
            e = np.exp(raw_logits - np.max(raw_logits))
            p = (e / np.sum(e)).tolist()
            conf = 1.0 if k < 2 else 1.0 - (-sum(x * math.log(max(x, 1e-12)) for x in p)) / math.log(k)
            if it["qtype"] == "choice":
                keys = list(it["crit"].keys())
                answers[qid] = {
                    "choice": keys[int(np.argmax(p))],
                    "probabilities": {keys[j]: round(p[j], 4) for j in range(len(keys))},
                    "confidence": round(conf, 4)
                }
            elif it["qtype"] == "score":
                answers[qid] = {
                    "score": round(sum(j * p[j] for j in range(len(p))), 4),
                    "confidence": round(conf, 4)
                }
            else:
                answers[qid] = {"noul": round(p[1], 4)}
        return answers


# ==========================================
# 2. Snake Game Logic
# ==========================================
class SnakeGame:
    DELTAS = {"UP": (0, -1), "DOWN": (0, 1), "LEFT": (-1, 0), "RIGHT": (1, 0)}
    OPPOSITES = {"UP": "DOWN", "DOWN": "UP", "LEFT": "RIGHT", "RIGHT": "LEFT"}

    def __init__(self, width=8, height=8):
        self.width = width
        self.height = height
        self.reset()

    def reset(self):
        mid_x, mid_y = self.width // 2, self.height // 2
        self.snake = [(mid_x, mid_y), (mid_x - 1, mid_y), (mid_x - 2, mid_y)]
        self.score = 0
        self.steps = 0
        self.alive = True
        self.last_direction = "RIGHT"
        self.food = self._spawn_food()
        return self

    def _spawn_food(self):
        empty = [(x, y) for y in range(self.height) for x in range(self.width) if (x, y) not in self.snake]
        return random.choice(empty) if empty else (-1, -1)

    def is_collision(self, pt):
        x, y = pt
        if x < 0 or x >= self.width or y < 0 or y >= self.height:
            return True
        return pt in self.snake[:-1]

    def get_perceptual_state(self):
        hx, hy = self.snake[0]
        fx, fy = self.food
        analysis = {}
        for d, (dx, dy) in self.DELTAS.items():
            nxt = (hx + dx, hy + dy)
            collides = self.is_collision(nxt)
            is_reverse = (d == self.OPPOSITES[self.last_direction]) and len(self.snake) > 1
            dist = abs(nxt[0] - fx) + abs(nxt[1] - fy)
            if collides or is_reverse:
                analysis[d] = f"DEADLY: {'reverse' if is_reverse else 'collision'}"
            else:
                analysis[d] = f"safe path, distance to food = {dist}"
        return {
            "game": "Snake",
            "grid": f"{self.width}x{self.height}",
            "head": [hx, hy],
            "food": [fx, fy],
            "steps": self.steps,
            "score": self.score,
            "options_analysis": analysis
        }

    def step(self, direction):
        if not self.alive:
            return False, False
        dx, dy = self.DELTAS[direction]
        hx, hy = self.snake[0]
        new_head = (hx + dx, hy + dy)
        if self.is_collision(new_head):
            self.alive = False
            return False, False
        self.snake.insert(0, new_head)
        self.last_direction = direction
        self.steps += 1
        ate_food = (new_head == self.food)
        if ate_food:
            self.score += 1
            self.food = self._spawn_food()
        else:
            self.snake.pop()
        return True, ate_food


# ==========================================
# 3. HTML/SVG UI Generator (Zero Flicker)
# ==========================================
def render_board_html(game, answers=None, latency_ms=0.0):
    cell_size = 40
    w_px = game.width * cell_size
    h_px = game.height * cell_size

    grid_cells = ""
    for y in range(game.height):
        for x in range(game.width):
            pt = (x, y)
            left = x * cell_size
            top = y * cell_size
            if pt == game.snake[0]:
                bg = "#10b981" if game.alive else "#ef4444"
                content = "👀" if game.alive else "💥"
                border_radius = "10px"
                shadow = "box-shadow: 0 0 12px rgba(16, 185, 129, 0.6);" if game.alive else ""
            elif pt in game.snake:
                bg = "linear-gradient(135deg, #34d399, #059669)"
                content = ""
                border_radius = "8px"
                shadow = ""
            elif pt == game.food:
                bg = "radial-gradient(circle, #f87171, #dc2626)"
                content = "🍎"
                border_radius = "50%"
                shadow = "box-shadow: 0 0 14px rgba(239, 68, 68, 0.8);"
            else:
                bg = "#1e293b"
                content = ""
                border_radius = "6px"
                shadow = ""

            grid_cells += f"""
            <div style="position: absolute; left: {left}px; top: {top}px; width: {cell_size - 4}px; height: {cell_size - 4}px;
                        background: {bg}; border-radius: {border_radius}; display: flex; align-items: center; justify-content: center;
                        font-size: 20px; user-select: none; transition: all 0.15s ease-in-out; {shadow}">
                {content}
            </div>
            """

    if answers:
        choice = answers["next_move"]["choice"]
        conf = answers["next_move"]["confidence"] * 100
        probs = answers["next_move"]["probabilities"]
        danger = answers["danger_level"]["score"]
        viability = answers["viable_path"]["noul"] * 100

        prob_bars = ""
        for d in ["UP", "DOWN", "LEFT", "RIGHT"]:
            p = probs.get(d, 0.0) * 100
            is_best = (d == choice)
            bar_color = "#10b981" if is_best else "#475569"
            badge = "<span style='color: #4ade80; font-weight: bold;'>★</span>" if is_best else ""
            prob_bars += f"""
            <div style="margin-bottom: 6px;">
                <div style="display: flex; justify-content: space-between; font-size: 13px; font-weight: 600; color: #f1f5f9;">
                    <span>{badge} {d}</span>
                    <span>{p:.1f}%</span>
                </div>
                <div style="background: #334155; height: 7px; border-radius: 4px; overflow: hidden; margin-top: 2px;">
                    <div style="width: {p}%; height: 100%; background: {bar_color}; transition: width 0.2s ease;"></div>
                </div>
            </div>
            """
        status_badge = "<span style='background: #065f46; color: #34d399; padding: 4px 10px; border-radius: 9999px; font-size: 13px; font-weight: 600;'>🟢 存活中</span>" if game.alive else "<span style='background: #991b1b; color: #fca5a5; padding: 4px 10px; border-radius: 9999px; font-size: 13px; font-weight: 600;'>💥 遊戲結束</span>"
    else:
        choice = "等待中"
        conf = 0
        danger = 0
        viability = 100
        prob_bars = "<div style='color: #94a3b8; font-size: 13px;'>點擊下方按鈕開始模擬</div>"
        status_badge = "<span style='background: #334155; color: #94a3b8; padding: 4px 10px; border-radius: 9999px; font-size: 13px;'>⏳ 就緒</span>"

    html = f"""
    <div style="background: #0f172a; border-radius: 16px; padding: 20px; color: #f8fafc; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; box-shadow: 0 10px 25px -5px rgba(0, 0, 0, 0.4); max-width: 800px; margin: 0 auto;">
        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 16px; border-bottom: 1px solid #334155; padding-bottom: 12px;">
            <div>
                <h2 style="margin: 0; font-size: 20px; color: #38bdf8; display: flex; align-items: center; gap: 8px;">
                    🐍 Laya System-1 貪吃蛇決策儀表板
                </h2>
                <div style="font-size: 12px; color: #94a3b8; margin-top: 3px;">無自回歸延遲 • 單次 Forward Pass 預測全決策</div>
            </div>
            <div>{status_badge}</div>
        </div>

        <div style="display: flex; gap: 24px; align-items: flex-start; flex-wrap: wrap;">
            <div style="position: relative; width: {w_px}px; height: {h_px}px; background: #0b1120; border-radius: 12px; padding: 2px; border: 2px solid #334155;">
                {grid_cells}
            </div>

            <div style="flex: 1; min-width: 260px; display: flex; flex-direction: column; gap: 12px;">
                <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 10px;">
                    <div style="background: #1e293b; padding: 10px; border-radius: 8px; border-left: 4px solid #38bdf8;">
                        <div style="font-size: 12px; color: #94a3b8;">當前得分</div>
                        <div style="font-size: 22px; font-weight: 700; color: #fbbf24;">🍎 {game.score}</div>
                    </div>
                    <div style="background: #1e293b; padding: 10px; border-radius: 8px; border-left: 4px solid #a855f7;">
                        <div style="font-size: 12px; color: #94a3b8;">存活步數</div>
                        <div style="font-size: 22px; font-weight: 700; color: #c084fc;">👣 {game.steps}</div>
                    </div>
                </div>

                <div style="background: #1e293b; padding: 14px; border-radius: 12px; border: 1px solid #334155;">
                    <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 8px;">
                        <span style="font-size: 13px; color: #94a3b8; font-weight: 600;">⚡ System-1 即時決策</span>
                        <span style="font-size: 11px; background: #0284c7; color: #e0f2fe; padding: 2px 8px; border-radius: 6px;">{latency_ms:.1f} ms</span>
                    </div>
                    <div style="font-size: 15px; margin-bottom: 10px;">
                        方向: <b style="color: #4ade80; font-size: 18px;">{choice}</b> 
                        <span style="font-size: 12px; color: #94a3b8; margin-left: 6px;">(信心度: {conf:.1f}%)</span>
                    </div>
                    <div style="display: flex; justify-content: space-between; font-size: 12px; color: #cbd5e1; margin-bottom: 10px; background: #0f172a; padding: 6px 10px; border-radius: 8px;">
                        <span>危險指數: <b>{danger:.2f}/3.0</b></span>
                        <span>路徑可行性: <b>{viability:.1f}%</b></span>
                    </div>
                    <div style="font-size: 12px; color: #94a3b8; margin-bottom: 6px; font-weight: 600;">各方向機率分佈 (Softmax):</div>
                    {prob_bars}
                </div>
            </div>
        </div>
    </div>
    """
    return html


# ==========================================
# 4. Gradio Interface Construction
# ==========================================
class ControlState:
    is_playing = False

def build_gradio_app():
    print("📥 載入 Laya 模型中...")
    model_dir = snapshot_download("receptron/laya-onnx", allow_patterns=["laya.onnx", "laya.onnx.data", "laya_config.json", "tokenizer/*"])
    agent = LayaAgent(model_dir)
    print("✅ Laya 模型準備就緒！")

    game = SnakeGame(8, 8)
    ctrl = ControlState()

    has_timer = hasattr(gr, "Timer")

    with gr.Blocks(title="Laya 貪吃蛇 - System-1 Decision Agent") as demo:
        board_display = gr.HTML(value=render_board_html(game))

        with gr.Row():
            btn_start = gr.Button("▶ 開始自動遊玩 (Auto Play)", variant="primary", scale=2)
            btn_step = gr.Button("⏭ 單步決策 (Step)", variant="secondary", scale=1)
            btn_pause = gr.Button("⏸ 暫停 (Pause)", scale=1)
            btn_reset = gr.Button("🔄 重新開局 (Reset)", scale=1)

        with gr.Row():
            speed_slider = gr.Slider(minimum=0.1, maximum=1.0, value=0.35, step=0.05, label="⏱ 步進間隔速度 (秒)")

        with gr.Accordion("🔍 檢視 Laya 模型輸入與原始輸出 (Debug State)", open=False):
            state_json = gr.JSON(label="最新感知狀態與決策結果")

        def do_one_step():
            if not game.alive:
                return render_board_html(game), {"status": "Game Over"}
            state = game.get_perceptual_state()
            t0 = time.perf_counter()
            answers = agent.system_one(state, {
                "next_move": {
                    "type": "choice",
                    "instructions": "Pick the safest direction towards food and away from obstacles.",
                    "criteria": state["options_analysis"]
                },
                "danger_level": {
                    "type": "score",
                    "instructions": "Assess current collision danger.",
                    "criteria": ["safe", "caution", "danger", "deadly"]
                },
                "viable_path": {
                    "type": "noul",
                    "instructions": "Is there a safe viable path to advance?"
                }
            })
            latency = (time.perf_counter() - t0) * 1000
            mv = answers["next_move"]["choice"]
            game.step(mv)
            html = render_board_html(game, answers, latency)
            debug_info = {"perceptual_state": state, "laya_answers": answers, "latency_ms": latency}
            return html, debug_info

        def on_reset():
            ctrl.is_playing = False
            game.reset()
            return render_board_html(game), {"status": "Reset", "score": 0}

        def on_pause():
            ctrl.is_playing = False
            return render_board_html(game), {"status": "Paused"}

        # Playback controller
        class PlaybackState:
            is_running = False

        state_ctrl = PlaybackState()

        def auto_play_loop(delay):
            state_ctrl.is_running = True
            max_steps = 100
            step_count = 0
            while state_ctrl.is_running and game.alive and step_count < max_steps:
                html, debug_info = do_one_step()
                step_count += 1
                yield html, debug_info
                time.sleep(delay)
            state_ctrl.is_running = False
            yield render_board_html(game), {"status": "Stopped / Game Over"}

        def stop_play():
            state_ctrl.is_running = False
            return render_board_html(game), {"status": "Paused"}

        def reset_play():
            state_ctrl.is_running = False
            game.reset()
            return render_board_html(game), {"status": "Reset", "score": 0}

        # Event bindings using native Gradio cancels
        play_event = btn_start.click(
            fn=auto_play_loop,
            inputs=[speed_slider],
            outputs=[board_display, state_json]
        )
        btn_pause.click(fn=stop_play, outputs=[board_display, state_json], cancels=[play_event])
        btn_reset.click(fn=reset_play, outputs=[board_display, state_json], cancels=[play_event])
        btn_step.click(fn=do_one_step, outputs=[board_display, state_json])

    # Enable queue with concurrency limit to prevent proxy timeouts
    demo.queue(default_concurrency_limit=5)
    return demo


def start_cloudflare_tunnel(port=7860):
    try:
        import subprocess, re, time
        res = subprocess.run(["which", "cloudflared"], capture_output=True, text=True)
        if res.returncode != 0:
            print("📦 正在自動安裝 Cloudflare Tunnel (徹底解決 504 逾時問題)...")
            subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"], check=False)
            subprocess.run(["dpkg", "-i", "cloudflared-linux-amd64.deb"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)

        proc = subprocess.Popen(
            ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True
        )
        cf_url = None
        start_t = time.time()
        while time.time() - start_t < 10:
            line = proc.stdout.readline()
            if not line:
                break
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if m:
                cf_url = m.group(0)
                break
            time.sleep(0.1)
        return cf_url, proc
    except Exception:
        return None, None


def launch_in_colab():
    app = build_gradio_app()
    
    colab_url = None
    cf_url = None
    
    # 1. Try Google Colab Native Proxy Port
    try:
        from google.colab.output import eval_js
        colab_url = eval_js("google.colab.kernel.proxyPort(7860)")
    except Exception:
        pass

    # 2. Try Cloudflare Tunnel
    try:
        cf_url, _ = start_cloudflare_tunnel(7860)
    except Exception:
        pass

    print("\n" + "=" * 70)
    print("🚀 【Laya 貪吃蛇 Web UI 啟動成功！】")
    print("💡 為避免 gradio.live 官方反向代理在海外發生 504 Gateway Time-out，")
    print("   請優先使用以下穩定通道：\n")
    if colab_url:
        print(f"👉 【1. Colab 原生直連（最推薦、零逾時）】: {colab_url}")
    if cf_url:
        print(f"👉 【2. Cloudflare 公開網址（手機/外網極速開啟）】: {cf_url}")
    print("👉 【3. 內嵌畫面】: 直接操作 Colab 程式碼儲存格下方的內嵌介面")
    print("=" * 70 + "\n")

    # In Colab, share=False avoids generating the unreliable gradio.live frpc link
    # Using server_name='0.0.0.0', inline=True, debug=True
    app.launch(share=False, inline=True, server_name="0.0.0.0", server_port=7860, debug=True)


if __name__ == "__main__":
    launch_in_colab()


In [ ]:
# 💡【備用高速隧道（可選）】：若您想在手機/外網開啟，且發現 gradio.live 偶爾壅塞，
# 可以執行此 Cell 建立極速 Cloudflare Tunnel，完全免費且零 504 逾時：
# !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null
# !cloudflared tunnel --url http://127.0.0.1:7860


---
## 🚀 Option 2: 透過 Node.js / TypeScript 執行 Child-pi/laya

Google Colab 內建 Node.js 環境。可以直接 Clone 儲存庫並透過 `npx tsx` 執行 TypeScript 版本的決策模擬。

In [ ]:
# Clone Child-pi/laya 儲存庫並安裝依賴
!git clone https://github.com/Child-pi/laya.git
%cd laya
!npm install
!npm install -g tsx

In [ ]:
# 執行 TypeScript 貪吃蛇 Demo
!npx tsx examples/snake_demo.ts

---
## 💡 Key Insights: 為什麼 Laya 適合遊戲決策與即時系統？

1. **極低延遲 (< 40ms)**：
   一般文字生成 LLM 生成 100 個 Token 需 2~5 秒，無法應用於即時物理反饋或遊戲迴圈。Laya 採 Non-Autoregressive 架構，一次 Forward Pass 即可輸出全部 typed answers。
2. **語意與感知的融合**：
   狀態中的文字描述（如 `safe path, distance = 3` vs `DEADLY: collision`）透過 ModernBERT 雙向注意力機制互相比較，精確壓制致命移動並獎勵向目標靠近的方向。
3. **雙系統架構 (Dual System)**：
   Laya 作為 System-1 負責高吞吐率、本能式的即時反應。當信心度低於閾值（如 `confidence < 0.4`）時，再呼叫較慢的 System-2（如 A* 尋路演算法或大型語言模型）進行深度規劃。